# B-1 · 读报错与提问：自学者的保命技能

> 学前缓冲带第 2 周，排在 B00（命令行）之后、B01（Python 语法）之前。
> 自学者没有助教坐在旁边，**报错信息就是程序在跟你说话**——学会读它，
> 你就拥有了整门课最重要的元技能。本周不需要会写 Python，只需要会「读」。

## 学习目标

学完本 notebook，你应该能够：

1. 说清楚 traceback（调用栈）的结构，并按正确顺序（**从下往上**）读它；
2. 认出 5 类高频错误：NameError、TypeError、IndexError、SyntaxError、FileNotFoundError；
3. 按「定位行号 → 打印中间变量 → 最小化复现」的流程排查一个陌生报错；
4. 按模板写出一段别人（或 AI）能直接帮你的提问。

> 先修要求：B00 的终端基础（会运行 `python`、知道文件路径是什么）。

## 1. 报错不是失败，是线索

新手看到红字的第一反应是「我搞砸了」，工程师的反应是「**它在告诉我哪一行、为什么**」。
Python 的报错（traceback）信息量极大，但排版对新手不友好——它是**倒着写的**：
真正的原因在**最后一行**，前面是「电话记录」（调用链）。

一次排查的三板斧（本周反复练习）：

1. **直接跳到最后一行**：看错误类型（`XxxError`）和那句英文描述；
2. **往上找第一处「你自己的代码」**：调用链里可能有库文件的行，你改不了也不用看，
   找到属于你文件的那一行——`File "你的文件.py", line 行号`；
3. **回到那一行盯着看**：80% 的错误在这一步就能看出原因。

In [1]:
# 解剖一个真实 traceback（这段代码故意写错）
import traceback


def read_sensor():
    data = [0.1, 0.2, 0.3]  # 传感器读数列表，只有 3 个元素
    return data[5]  # ← 错在这里：取第 6 个元素


def compute_average():
    value = read_sensor()  # 调用链的上一层
    return value / 1.0


try:
    compute_average()
except Exception:
    traceback.print_exc()

Traceback (most recent call last):
  File "/tmp/ipykernel_1665919/224592346.py", line 16, in <module>
    compute_average()
  File "/tmp/ipykernel_1665919/224592346.py", line 11, in compute_average
    value = read_sensor()  # 调用链的上一层
            ^^^^^^^^^^^^^
  File "/tmp/ipykernel_1665919/224592346.py", line 7, in read_sensor
    return data[5]  # ← 错在这里：取第 6 个元素
           ~~~~^^^
IndexError: list index out of range


把上面的输出**从下往上**读：

```
IndexError: list index out of range          ← 最后一行：什么错（列表下标越界）
File "...ipynb", line 7, in read_sensor      ← 你的代码：哪个文件、第几行、哪个函数
    return data[5]                           ← 出错的那行原样摆出来
File "...ipynb", line 11, in compute_average ← 谁调用了它（电话记录，逐层往上）
```

**错误类型 + 英文描述 = 搜索引擎/AI 的最佳输入**。`IndexError: list index out of range`
原样丢给搜索引擎，第一条答案基本就是解法。

## 2. 五类高频错误逐个认

### 2.1 NameError —— 名字写错了

变量没定义就拿来用。九成是**拼写错误**或**大小写错误**（Python 区分大小写）。

In [2]:
temperature = 25.0

try:
    print(temprature)      # 少了一个 a
except NameError:
    traceback.print_exc()

Traceback (most recent call last):
  File "/tmp/ipykernel_1665919/3786058473.py", line 4, in <module>
    print(temprature)      # 少了一个 a
          ^^^^^^^^^^
NameError: name 'temprature' is not defined


读法：`name 'temprature' is not defined` = 「这个名字我不认识」。
**自查**：拼写？大小写？是不是定义在后面、用在了前面？

### 2.2 TypeError —— 类型不匹配

把两种不能混用的东西放在一起运算，最经典的是「字符串 + 数字」。

In [3]:
temperature = 25.0

try:
    message = "当前温度: " + temperature   # 字符串不能直接加浮点数
except TypeError:
    traceback.print_exc()

Traceback (most recent call last):
  File "/tmp/ipykernel_1665919/610775970.py", line 4, in <module>
    message = "当前温度: " + temperature   # 字符串不能直接加浮点数
              ~~~~~~~~~~~~~^~~~~~~~~~~~~
TypeError: can only concatenate str (not "float") to str


读法：`can only concatenate str (not "float") to str` = 「字符串只能和字符串拼接」。
**自查**：用 `type(变量)` 打印看看它到底是什么类型。修法：`f"当前温度: {temperature}"`（B01 会学）。

### 2.3 IndexError —— 下标越界

列表只有 N 个元素，合法下标是 `0` 到 `N-1`。**数人头从 0 开始**是新手最容易摔的地方。

In [4]:
readings = [0.1, 0.2, 0.3]   # 3 个元素，合法下标：0, 1, 2

try:
    print(readings[3])
except IndexError:
    traceback.print_exc()

Traceback (most recent call last):
  File "/tmp/ipykernel_1665919/215971782.py", line 4, in <module>
    print(readings[3])
          ~~~~~~~~^^^
IndexError: list index out of range


### 2.4 SyntaxError —— 句子本身写错了

前面三类是「程序跑起来才错」，SyntaxError 是「**还没跑就错了**」——
代码本身不符合语法，Python 连读都读不下去。括号不配对、冒号漏了、缩进错了都是这一类。

In [5]:
bad_code = "def f(:\n    pass\n"   # 括号都没配对

try:
    exec(bad_code)        # exec 把字符串当代码执行，用来演示
except SyntaxError:
    traceback.print_exc()

Traceback (most recent call last):
  File "/tmp/ipykernel_1665919/1276871512.py", line 4, in <module>
    exec(bad_code)        # exec 把字符串当代码执行，用来演示
    ^^^^^^^^^^^^^^
  File "<string>", line 1
    def f(:
          ^
SyntaxError: invalid syntax


读法：`invalid syntax` + 一个指向出错位置的 `^` 箭头。
**自查**：报错行的**上一行**也要看——括号没配对时，Python 常常在下一行才发现。

### 2.5 FileNotFoundError —— 路径不对

要读的文件不在你以为的地方。九成的根源是 B00 讲过的**相对路径 vs 当前工作目录**。

In [6]:
try:
    with open("data/实验数据.csv", encoding="utf-8") as f:
        print(f.read())
except FileNotFoundError:
    traceback.print_exc()

Traceback (most recent call last):
  File "/tmp/ipykernel_1665919/3107460872.py", line 2, in <module>
    with open("data/实验数据.csv", encoding="utf-8") as f:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/data/wangf/robot_rl_learn/.venv/lib/python3.11/site-packages/IPython/core/interactiveshell.py", line 350, in _modified_open
    return io_open(file, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: 'data/实验数据.csv'


读法：`No such file or directory: 'data/实验数据.csv'`。
**自查**：`pwd` 看当前在哪、`ls` 看文件实际在哪，路径是相对谁的？（回 B00 场景 1–3 复习。）

## 3. 通用排查流程：四步收尾

遇到没见过的报错也别慌，按顺序来：

1. **读最后一行**，把错误类型+描述复制下来；
2. **定位你的行号**，回到那行代码；
3. **打印中间变量**：在出错行前加 `print(变量)` 或 `print(type(变量))`，
   看看实际值和你的想象是否一致——90% 的 bug 是「你以为它是 A，其实它是 B」；
4. **最小化复现**：把代码删到只剩能触发错误的最短片段（通常 5 行以内）。
   这一步做完，你往往自己就看出问题了；看不出，这段最小代码就是提问的材料。

## 4. 提问的艺术：让别人能帮你

自学不等于孤军奋战——搜索引擎、论坛、AI 都是你的助教，但前提是**你会问**。

**反例**（没人能帮你）：「我的代码报错了，怎么办？在线等！」

**正例模板**（五要素）：

```
【环境】Windows 11 + Python 3.11（或：课程仓库 .venv）
【我在做什么】B03 练习 2，想给矩阵做乘法
【最小复现代码】（5 行以内，别人复制就能跑）
import numpy as np
a = np.array([1, 2])
b = np.array([[1, 2], [3, 4]])
print(a * b)
【完整报错】（从 traceback 第一行到最后一行，完整复制，不要截图裁剪）
【我试过】把 b 改成一维，报错变了但还不对；搜了 "numpy matmul shape" 没看懂
```

> 要点：**最小代码 + 完整报错 + 已尝试**。给 AI 提问时把模板原样粘贴即可；
> 这也是本课程 `journal/troubleshooting.md` 错题本的记录格式。

## ✏️ 练习

**练习 1（20 分，难度 ★）读一段陌生 traceback**
不运行，仅凭阅读回答：下面这段报错是什么错误类型？出在第几行？最可能的原因是什么？

```
Traceback (most recent call last):
  File "train.py", line 42, in <module>
    rewards = load_rewards("runs/w05/reward.txt")
  File "train.py", line 17, in load_rewards
    with open(path) as f:
FileNotFoundError: [Errno 2] No such file or directory: 'runs/w05/reward.txt'
```

**练习 2（25 分，难度 ★★）修三个小程序**
下面三段各有一个错误。先说出错误类型，再改正（在你的环境里运行验证）：

```python
# 程序 A：想打印 3 次 "训练开始"
for i in range(3)
    print("训练开始")

# 程序 B：想计算奖励平均值
rewards = [10.0, 20.0, 30.0]
average = sum(rewards) / len(rewards) + bonus

# 程序 C：想读取配置里的学习率
config = {"learning_rate": 0.001, "gamma": 0.99}
print(config["learing_rate"])
```

**练习 3（25 分，难度 ★★）最小化复现**
助教收到一段 40 行的报错程序。用第 3 节的流程说明：
你会按什么顺序把它压缩到 5 行以内？每一步在排除什么？

**练习 4（30 分，难度 ★★★）写一条合格提问**
假设你在 B05 练习中遇到 `TypeError: 'numpy.float64' object is not callable`，
按第 4 节的五要素模板，写出一条可以发给 AI 或论坛的完整提问
（环境、最小代码可以虚构，但要合理、自洽）。

<details>
<summary>👉 参考答案（先独立做，再点开）</summary>

**练习 1**
错误类型：`FileNotFoundError`。出错行：`train.py` 第 17 行（`open(path)` 这一行），
由第 42 行调用传入路径 `'runs/w05/reward.txt'`。
最可能的原因：运行 `train.py` 时的**当前工作目录**不在项目根目录
（相对路径解析错了），或者文件名/目录名拼写与磁盘上的不一致。

**练习 2**
- 程序 A：`SyntaxError`——`for` 语句末尾漏了冒号，改为 `for i in range(3):`；
- 程序 B：`NameError`——`bonus` 没定义就使用，先定义（如 `bonus = 5.0`）或从算式中去掉；
- 程序 C：`KeyError`——字典里没有 `"learing_rate"` 这个键（`learning` 拼成了 `learing`），
  改为 `config["learning_rate"]`。注意它不是本周讲的五类之一，
  但读法完全相同：最后一行会告诉你 `KeyError: 'learing_rate'`。

**练习 3**
参考步骤：① 先备份原文件；② 删掉与报错调用链无关的函数和分支，
每删一批就重跑一次，**报错仍在就继续删**；③ 报错一旦消失，说明刚删掉的部分里有元凶，
恢复回来换一批删；④ 最终剩下「触发同一报错的最短代码」。
每步都在排除「无关代码的干扰」，同时验证问题是否依然复现。

**练习 4**
示例（合理自洽即可）：

```
【环境】Ubuntu 22.04 + 课程仓库 .venv（Python 3.11，numpy 2.x）
【我在做什么】B05 练习 1，用 solve_ivp 仿真一阶环节，想给时间常数 tau 加倍
【最小复现代码】
import numpy as np
tau = np.float64(2.0)
t = tau(0.5)        # 我本以为 tau 是函数，其实它是数
【完整报错】TypeError: 'numpy.float64' object is not callable
【我试过】把 tau 换成 int 也一样报错；搜了错误信息，
          答案说"把变量当函数调用了"，但我没找到在哪调用的——第 2 行就是。
```
</details>

## 延伸阅读

- [Python 官方教程：错误与异常（中文）](https://docs.python.org/zh-cn/3/tutorial/errors.html)——B02 学异常处理前的最佳预热；
- [《提问的智慧》中文译本](https://github.com/ryanhanwu/How-To-Ask-Questions-The-Smart-Way/blob/main/README-zh_CN.md)——很长，只读「如何描述问题」两节即可；
- [Real Python: Understanding the Python Traceback](https://realpython.com/python-traceback/)——traceback 各部分的逐行拆解（英文）；
- 本课程约定：把你实际遇到的报错按第 4 节模板记入 `journal/troubleshooting.md`，
  期末它就是你最有价值的资产。

## 小结

- traceback **从下往上读**：最后一行是「什么错」，往上找你**自己代码**的行号；
- 五类常客：NameError（拼写）、TypeError（类型）、IndexError（越界）、
  SyntaxError（语法，看上一行）、FileNotFoundError（路径，想相对/绝对）；
- 排查四步：读最后一行 → 定位行号 → 打印中间变量 → 最小化复现；
- 提问五要素：环境 / 在做什么 / 最小代码 / 完整报错 / 已尝试；
- 下一周 B00 学命令行时，每敲错一条命令就练习一次「读报错」——技能是摔出来的。